# PARKIT 2차 학습 (M1~M5)

17 문서 2절 순서대로 씁니다.

1. 구글 드라이브 **내 드라이브** 맨 위에 `split_2차.zip`을 올립니다(PC의 `Parkit\dataset\split_2차.zip`).
2. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**로 바꾸고 저장합니다.
3. **런타임 → 모두 실행**을 누르거나, 위에서부터 셀을 하나씩 실행합니다.

학습한 모델은 끝나는 즉시 드라이브 `parkit_runs_2차/M1` ~ `M5`에 `best.pt`, `results.png`, `results.csv`, `args.yaml`이 저장됩니다.
연결이 끊기면 셀을 처음부터 다시 실행하면 됩니다. 드라이브에 `best.pt`가 이미 있는 모델은 건너뛰고, 끝나지 않은 모델은 처음부터 다시 학습합니다.

## 1. GPU 확인과 설치

In [ ]:
!nvidia-smi
!pip install -q ultralytics

## 2. 설정 (필요하면 여기만 고치기)

In [ ]:
DRIVE = '/content/drive/MyDrive'
ZIP = f'{DRIVE}/split_2차.zip'      # 드라이브에 올린 압축 파일
SAVE = f'{DRIVE}/parkit_runs_2차'    # 결과를 저장할 드라이브 폴더
DATA = '/content/split_2차'          # 압축을 풀 곳 (코랩 안, 빠름)
MODELS = ['M1', 'M2', 'M3', 'M4', 'M5']  # 차례로 학습할 모델
# 지난 학습과 같은 설정. 모든 모델에 똑같이 쓴다
TRAIN_ARGS = dict(model='yolo26n.pt', imgsz=640, epochs=100, batch=16, hsv_h=0.05, seed=0, deterministic=True)

## 3. 드라이브 연결, 압축 풀기

드라이브 접근 허용 창이 뜨면 허용합니다. 드라이브에서 바로 읽으면 느리므로 코랩 안(`/content`)에 풀어서 씁니다.

In [ ]:
import os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(ZIP):
    raise FileNotFoundError(f'{ZIP} 이 없습니다. 내 드라이브 맨 위에 split_2차.zip을 올렸는지 확인하세요.')
if not os.path.isdir(DATA):
    shutil.copy(ZIP, '/content/split_2차.zip')
    with zipfile.ZipFile('/content/split_2차.zip') as zf:  # 한글 파일명도 그대로 풀린다
        zf.extractall('/content')

for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{DATA}/images/{s}')), '장')
for m in MODELS:
    print(m, 'train', sum(1 for l in open(f'{DATA}/train_{m}.txt') if l.strip()), '장')
print(open(f'{DATA}/확인결과.txt', encoding='utf-8').read().split('== 나누기별')[0])

## 4. M1~M5 차례로 학습

모델 하나가 끝날 때마다 바로 드라이브에 저장합니다. 드라이브에 `best.pt`가 있는 모델은 건너뜁니다.
예상 시간(추정, T4): M1 약 5분, M5 약 20분, 다섯 개 합쳐 1시간 안팎. 학습 중에는 브라우저 탭을 열어 두고 PC가 절전 모드로 가지 않게 합니다.

In [ ]:
import os, shutil, time
from ultralytics import YOLO

os.makedirs(SAVE, exist_ok=True)
for m in MODELS:
    dst = f'{SAVE}/{m}'
    if os.path.exists(f'{dst}/best.pt'):
        print(f'{m}: 드라이브에 best.pt가 있어 건너뜀')
        continue
    print(f'===== {m} 학습 시작 =====')
    t0 = time.time()
    args = dict(TRAIN_ARGS)
    model = YOLO(args.pop('model'))
    model.train(data=f'{DATA}/data_{m}.yaml', project='/content/runs', name=m, exist_ok=True, **args)

    run = f'/content/runs/{m}'
    os.makedirs(dst, exist_ok=True)
    for f in ('results.png', 'results.csv', 'args.yaml'):
        if os.path.exists(f'{run}/{f}'):
            shutil.copy(f'{run}/{f}', f'{dst}/{f}')
    # best.pt는 마지막에, 임시 이름으로 복사한 뒤 바꾼다(best.pt가 있으면 '끝난 모델'로 보므로)
    shutil.copy(f'{run}/weights/best.pt', f'{dst}/best.pt.part')
    os.replace(f'{dst}/best.pt.part', f'{dst}/best.pt')
    print(f'{m}: 끝 ({(time.time() - t0) / 60:.1f}분) → {dst}')

## 5. 저장된 결과 확인

다섯 모델 모두 `best.pt`가 있으면 끝입니다. 드라이브의 `parkit_runs_2차` 폴더를 PC로 내려받고, 채점은 PC에서 `eval_2차.py`로 합니다(17 문서 3절).

In [ ]:
import os
for m in MODELS:
    d = f'{SAVE}/{m}'
    files = sorted(os.listdir(d)) if os.path.isdir(d) else []
    print(m, '완료' if 'best.pt' in files else '미완료', files)

## 6. (선택) results.png 보기

In [ ]:
from IPython.display import Image, display
import os
for m in MODELS:
    p = f'{SAVE}/{m}/results.png'
    if os.path.exists(p):
        print(m)
        display(Image(p, width=900))